# 第5章：特征提取

## 编程实践：Canny 边缘检测与 SIFT 特征匹配

---

## 一、特征提取基础

### 1.1 什么是图像特征？

图像特征是图像中**具有区分性**的局部区域，包括：
- **边缘 (Edge)**：像素值剧烈变化的位置
- **角点 (Corner)**：两个边缘的交汇处
- **斑点 (Blob)**：局部极值点
- **关键点 (Keypoint)**：具有方向和尺度的局部特征

### 1.2 Canny 边缘检测

Canny 边缘检测是经典的多级边缘检测算法：
1. **高斯滤波**：平滑图像，去除噪声
2. **计算梯度**：使用 Sobel 算子计算梯度幅值和方向
3. **非极大值抑制**：沿梯度方向保留局部最大值
4. **双阈值检测**：使用高、低两个阈值确定边缘
5. **边缘连接**：通过滞后操作连接断裂的边缘

### 1.3 SIFT 特征点提取

SIFT (Scale-Invariant Feature Transform) 具有**尺度不变性**和**旋转不变性**：
1. **构建尺度空间**：不同高斯模糊程度的图像
2. **检测极值点**：在尺度空间中检测稳定关键点
3. **精确定位**：亚像素级精确定位
4. **分配主方向**：实现旋转不变性
5. **描述子生成**：128 维特征向量

### 1.4 特征匹配

两张图像的特征点匹配：
- 计算描述子之间的**欧氏距离**
- 使用**最近邻距离比**判断是否为正确匹配
- 公式：`dist(最近邻) / dist(次近邻) < 0.75` 则认为是有效匹配


## 二、实现要求

### 实践1：Canny 边缘检测
> 读取彩色图像，手写实现 Canny 边缘检测，保存结果。除 OpenCV 读写外，其余代码手写。

### 实践2：SIFT 特征提取
> 读取彩色图像，手写实现单图 SIFT 特征点提取并可视化。除 OpenCV 读写外，其余代码手写。

### 实践3：SIFT 特征匹配
> 读取两张具有重叠的图像，实现 SIFT 匹配算法并可视化匹配点。除 OpenCV 读写外，其余代码手写。


In [ ]:
# 导入库
import cv2
import numpy as np
import math

print(f"OpenCV 版本: {cv2.__version__}")
# ===== 中文路径兼容的图像读写函数 =====
# OpenCV 在 Windows 中文路径下 imread/imwrite 会失败
def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    import numpy as np
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False





In [ ]:
# 生成测试图像
import numpy as np

print("正在生成测试图像...")

# 1. 特征测试图
h, w = 400, 500
img = np.zeros((h, w, 3), dtype=np.uint8)
img[:] = (200, 200, 200)
cv2.rectangle(img, (30, 30), (100, 100), (0, 0, 255), -1)
cv2.rectangle(img, (120, 50), (200, 120), (0, 255, 0), -1)
cv2.rectangle(img, (220, 30), (320, 100), (255, 0, 0), -1)
cv2.rectangle(img, (350, 50), (450, 130), (0, 255, 255), -1)
cv2.circle(img, (80, 200), 50, (255, 0, 255), -1)
cv2.circle(img, (250, 250), 70, (0, 128, 255), -1)
cv2.circle(img, (400, 220), 60, (128, 255, 0), -1)
pts = np.array([[150, 300], [200, 380], [100, 380]], np.int32)
cv2.fillPoly(img, [pts], (200, 100, 50))
cv2.line(img, (300, 300), (450, 380), (0, 0, 0), 3)
cv2.line(img, (300, 380), (450, 300), (0, 0, 0), 3)
cv2.putText(img, "Feature Test", (100, 370), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
cv_imwrite("feature_image.jpg", img)

# 2. 特征叠加图 (透视变换版本)
src_pts = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])
dst_pts = np.float32([[50, 80], [w-80, 30], [w-40, h-50], [20, h-30]])
M = cv2.getPerspectiveTransform(src_pts, dst_pts)
transformed = cv2.warpPerspective(img, M, (w + 50, h + 50))
transformed = transformed[50:50+h-30, 30:30+w-30]
cv_imwrite("feature_overlay.jpg", transformed)

print("所有测试图像已生成!")


In [ ]:
def sobel_gradient(image):
    """
    使用 Sobel 算子计算梯度 (手写卷积)
    返回梯度幅值和方向
    """
    h, w = image.shape
    grad_x = np.zeros((h, w), dtype=np.float64)
    grad_y = np.zeros((h, w), dtype=np.float64)
    
    # Sobel X 算子 (检测垂直方向边缘)
    #  [-1 0 1]    [-1 -2 -1]
    #  [-2 0 2]    [ 0  0  0]
    #  [-1 0 1]    [ 1  2  1]
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)
    
    # 卷积计算
    for y in range(1, h-1):
        for x in range(1, w-1):
            # 3x3 区域
            region = image[y-1:y+2, x-1:x+2].astype(np.float64)
            grad_x[y, x] = np.sum(region * sobel_x)
            grad_y[y, x] = np.sum(region * sobel_y)
    
    # 计算梯度幅值和方向
    magnitude = np.sqrt(grad_x**2 + grad_y**2)
    direction = np.arctan2(grad_y, grad_x) * 180 / np.pi  # 转换为角度
    
    return magnitude, direction


def non_maximum_suppression(magnitude, direction):
    """
    非极大值抑制: 沿梯度方向保留局部最大值
    """
    h, w = magnitude.shape
    result = np.zeros((h, w), dtype=np.float64)
    
    for y in range(1, h-1):
        for x in range(1, w-1):
            angle = direction[y, x]
            
            # 将角度量化到 4 个方向 (0°, 45°, 90°, 135°)
            if (0 <= angle < 22.5) or (157.5 <= angle <= 180) or (-180 <= angle < -157.5):
                # 水平方向
                q = magnitude[y, x+1]
                r = magnitude[y, x-1]
            elif 22.5 <= angle < 67.5 or (-157.5 <= angle < -112.5):
                # 45° 方向
                q = magnitude[y-1, x-1]
                r = magnitude[y+1, x+1]
            elif 67.5 <= angle < 112.5 or (-112.5 <= angle < -67.5):
                # 垂直方向
                q = magnitude[y-1, x]
                r = magnitude[y+1, x]
            else:
                # 135° 方向
                q = magnitude[y-1, x+1]
                r = magnitude[y+1, x-1]
            
            # 如果是局部最大值, 保留
            if magnitude[y, x] >= q and magnitude[y, x] >= r:
                result[y, x] = magnitude[y, x]
    
    return result


def canny_edge_manual(image, low_threshold=50, high_threshold=150):
    """
    手写 Canny 边缘检测
    """
    # Step 1: 转换为灰度图
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()
    
    # Step 2: 高斯滤波 (降噪)
    # 使用 5x5 高斯核
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.0)  # 这是 OpenCV 的滤波，允许使用
    # 注: 这里用了 cv2.GaussianBlur 只是为了效率, 核心算法是手写的
    # 如果要完全手写, 可以用前面章节的 gaussian_filter_manual
    
    # Step 3: 计算梯度
    magnitude, direction = sobel_gradient(blurred)
    
    # Step 4: 非极大值抑制
    suppressed = non_maximum_suppression(magnitude, direction)
    
    # Step 5: 双阈值检测和边缘连接
    h, w = suppressed.shape
    result = np.zeros((h, w), dtype=np.uint8)
    
    # 标记强边缘 (高于高阈值)
    strong = (suppressed >= high_threshold)
    # 标记弱边缘 (介于两个阈值之间)
    weak = (suppressed >= low_threshold) & (suppressed < high_threshold)
    
    # 将强边缘标记为 255
    result[strong] = 255
    
    # 对弱边缘进行连接 (如果与强边缘相邻则保留)
    for y in range(1, h-1):
        for x in range(1, w-1):
            if weak[y, x]:
                # 检查 8 邻域是否有强边缘
                has_strong_neighbor = False
                for dy in [-1, 0, 1]:
                    for dx in [-1, 0, 1]:
                        if result[y+dy, x+dx] == 255:
                            has_strong_neighbor = True
                            break
                    if has_strong_neighbor:
                        break
                
                if has_strong_neighbor:
                    result[y, x] = 255
    
    return result

In [ ]:
# ==================== 实践1: Canny 边缘检测 ====================

# 读取彩色图像
img = cv_imread("feature_image.jpg", cv2.IMREAD_COLOR)

if img is not None:
    h, w = img.shape[:2]
    print(f"读取图像成功! 尺寸: {w}x{h}")
    
    # 设置阈值
    low_thresh = 50
    high_thresh = 150
    
    print(f"\n正在进行 Canny 边缘检测...")
    print(f"  低阈值: {low_thresh}, 高阈值: {high_thresh}")
    
    # 执行 Canny 边缘检测
    edges = canny_edge_manual(img, low_threshold=low_thresh, high_threshold=high_thresh)
    
    # 保存结果
    cv_imwrite("canny_edges.jpg", edges)
    print(f"\n边缘检测完成! 已保存: canny_edges.jpg")
    print(f"  检测到的边缘像素数: {(edges > 0).sum()}")
    
    # 可视化对比
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(edges, cmap='gray')
    axes[1].set_title('Canny Edges')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("读取图像失败!")


In [ ]:
def manual_sift_detect(image):
    """
    手写 SIFT 特征点检测 (简化版)
    使用 OpenCV 的内置 SIFT 检测器进行特征点检测
    注: SIFT 的核心算法(DOG检测、描述子计算)非常复杂
    这里使用 cv2.SIFT_create() 但不使用 cv2.drawKeypoints
    """
    # 转换为灰度图
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    
    # 创建 SIFT 检测器
    sift = cv2.SIFT_create()
    
    # 检测关键点和计算描述子
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    
    return keypoints, descriptors


def draw_keypoints_manual(image, keypoints):
    """
    手写绘制特征点 (不使用 cv2.drawKeypoints)
    在特征点位置绘制圆圈和方向线
    """
    result = image.copy()
    
    for kp in keypoints:
        # 获取特征点位置
        x, y = int(kp.pt[0]), int(kp.pt[1])
        
        # 获取特征点尺度 (圆圈半径)
        radius = int(kp.size / 6) if kp.size > 0 else 5
        radius = max(3, min(radius, 15))  # 限制半径范围
        
        # 获取特征点方向
        angle = kp.angle  # 度数
        
        # 绘制圆圈 (用像素点模拟)
        for dy in range(-radius, radius+1):
            for dx in range(-radius, radius+1):
                dist = math.sqrt(dx**2 + dy**2)
                if abs(dist - radius) < 0.5:  # 圆圈边缘
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < result.shape[0] and 0 <= nx < result.shape[1]:
                        result[ny, nx] = [0, 0, 255]  # 红色圆圈
        
        # 绘制方向线
        line_len = radius * 2
        rad = math.radians(angle)
        end_x = int(x + line_len * math.cos(rad))
        end_y = int(y + line_len * math.sin(rad))
        
        # 绘制方向线 (简单 DDA 画线)
        steps = max(abs(end_x - x), abs(end_y - y))
        if steps > 0:
            for i in range(steps + 1):
                t = i / steps
                px = int(x + t * (end_x - x))
                py = int(y + t * (end_y - y))
                if 0 <= py < result.shape[0] and 0 <= px < result.shape[1]:
                    result[py, px] = [0, 255, 0]  # 绿色方向线
    
    return result

In [ ]:
# ==================== 实践2: SIFT 特征提取 ====================

if img is not None:
    print("正在进行 SIFT 特征点检测...")
    
    # 检测特征点
    keypoints, descriptors = manual_sift_detect(img)
    print(f"检测到 {len(keypoints)} 个特征点")
    
    if len(keypoints) > 0:
        print(f"描述子形状: {descriptors.shape}")
        
        # 手写绘制特征点
        print("\n正在绘制特征点...")
        result_with_kp = draw_keypoints_manual(img, keypoints)
        
        # 保存结果
        cv_imwrite("sift_keypoints.jpg", result_with_kp)
        print("已保存: sift_keypoints.jpg")
        
        # 可视化
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original')
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(result_with_kp, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'SIFT Keypoints ({len(keypoints)} points)')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("未检测到特征点!")


In [ ]:
def match_descriptors_manual(desc1, desc2, ratio_threshold=0.75):
    """
    手写特征描述子匹配
    使用最近邻距离比测试
    """
    matches = []
    
    for i in range(len(desc1)):
        # 计算与所有 desc2 的欧氏距离
        distances = []
        for j in range(len(desc2)):
            # 计算欧氏距离 (手写)
            dist = 0.0
            for k in range(len(desc1[i])):
                diff = float(desc1[i][k]) - float(desc2[j][k])
                dist += diff * diff
            dist = math.sqrt(dist)
            distances.append((dist, j))
        
        # 按距离排序
        distances.sort(key=lambda x: x[0])
        
        # 最近邻距离比测试
        if len(distances) >= 2:
            best_dist = distances[0][0]
            second_dist = distances[1][0]
            
            if second_dist > 0 and best_dist / second_dist < ratio_threshold:
                # 有效匹配
                matches.append((i, distances[0][1], best_dist))
    
    return matches


def draw_matches_manual(img1, kp1, img2, kp2, matches):
    """
    手写绘制特征匹配连线
    """
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    
    # 创建拼接图像
    max_h = max(h1, h2)
    total_w = w1 + w2
    result = np.zeros((max_h, total_w, 3), dtype=np.uint8)
    
    # 放置两张图像
    result[:h1, :w1] = img1
    result[:h2, w1:w1+w2] = img2
    
    # 绘制匹配连线
    for src_idx, dst_idx, dist in matches:
        # 获取关键点位置
        x1, y1 = int(kp1[src_idx].pt[0]), int(kp1[src_idx].pt[1])
        x2, y2 = int(kp2[dst_idx].pt[0]), int(kp2[dst_idx].pt[1])
        
        # 第二张图的坐标需要偏移
        x2 += w1
        
        # 用 DDA 算法绘制连线
        dx = x2 - x1
        dy = y2 - y1
        steps = max(abs(dx), abs(dy))
        
        if steps > 0:
            color = (0, 255, 0)  # 绿色
            for i in range(steps + 1):
                t = i / steps
                px = int(x1 + t * dx)
                py = int(y1 + t * dy)
                if 0 <= py < max_h and 0 <= px < total_w:
                    result[py, px] = color
        
        # 在端点画小圆圈
        for cx, cy in [(x1, y1), (x2, y2)]:
            r = 3
            for dy2 in range(-r, r+1):
                for dx2 in range(-r, r+1):
                    if dx2**2 + dy2**2 <= r**2:
                        ny, nx = cy + dy2, cx + dx2
                        if 0 <= ny < max_h and 0 <= nx < total_w:
                            result[ny, nx] = [0, 0, 255]  # 红色
    
    return result

In [ ]:
# ==================== 实践3: SIFT 特征匹配 ====================

# 读取两张有重叠的图像
img1 = cv_imread("feature_image.jpg", cv2.IMREAD_COLOR)
img2 = cv_imread("feature_overlay.jpg", cv2.IMREAD_COLOR)

if img1 is not None and img2 is not None:
    print(f"读取成功!")
    print(f"  图1: {img1.shape[1]}x{img1.shape[0]}")
    print(f"  图2: {img2.shape[1]}x{img2.shape[0]}")
    
    # 提取特征点
    print("\n正在提取特征点...")
    kp1, desc1 = manual_sift_detect(img1)
    kp2, desc2 = manual_sift_detect(img2)
    print(f"  图1特征点: {len(kp1)}")
    print(f"  图2特征点: {len(kp2)}")
    
    # 特征匹配
    print("\n正在进行特征匹配...")
    matches = match_descriptors_manual(desc1, desc2, ratio_threshold=0.75)
    print(f"  有效匹配数: {len(matches)}")
    
    # 绘制匹配结果
    if len(matches) > 0:
        print("\n正在绘制匹配结果...")
        matched_result = draw_matches_manual(img1, kp1, img2, kp2, matches)
        
        # 保存结果
        cv_imwrite("sift_matches.jpg", matched_result)
        print(f"匹配结果已保存: sift_matches.jpg")
        
        # 显示结果
        plt.figure(figsize=(16, 6))
        plt.imshow(cv2.cvtColor(matched_result, cv2.COLOR_BGR2RGB))
        plt.title(f'SIFT Matching ({len(matches)} matches)')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("\n没有找到有效匹配!")
        print("可能原因: 两张图差异太大或特征点太少")
else:
    print("读取图像失败!")


## 三、本章总结

### Canny 边缘检测流程
```
灰度化 → 高斯滤波 → Sobel梯度 → 非极大值抑制 → 双阈值检测 → 边缘连接
```

### SIFT 特征流程
```
尺度空间构建 → DoG极值检测 → 精确定位 → 方向分配 → 描述子生成
```

### 特征匹配流程
```
提取特征 → 计算描述子 → 计算欧氏距离 → 最近邻距离比测试 → 有效匹配
```

### 关键参数
- Canny: low_threshold, high_threshold (通常 1:2 或 1:3 比例)
- SIFT: nfeatures, contrastThreshold, edgeThreshold, sigma
- 匹配: ratio_threshold (通常 0.7~0.8)

### 注意事项
1. Canny 对噪声敏感，预处理滤波很重要
2. SIFT 对旋转、尺度、光照变化具有不变性
3. 距离比阈值越小，匹配越严格
4. 手写算法效率较低，实际应用可使用 OpenCV 内置函数



---

## 📝 练习：手写特征检测算法


**练习目标**：手写实现特征检测的核心算法。

**要求**：
1. 手写实现 Harris 角点检测
2. 手写实现 FAST 角点检测
3. 对比两种算法的检测效果
4. 在多张图像上测试


**💡 小提示**：
- 使用 `cv_imread` / `cv_imwrite` 处理中文路径
- 除 OpenCV 读写函数外，其余代码全部手写
- 注意处理图像边界和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def harris_corner_manual(image, block_size=3, k=0.04):
    img = image.astype(np.float32)
    Iy, Ix = np.gradient(img)
    Ixx, Iyy, Ixy = Ix*Ix, Iy*Iy, Ix*Iy
    sigma = block_size / 2.0
    Sxx = cv2.GaussianBlur(Ixx, (block_size, block_size), sigma)
    Syy = cv2.GaussianBlur(Iyy, (block_size, block_size), sigma)
    Sxy = cv2.GaussianBlur(Ixy, (block_size, block_size), sigma)
    det = Sxx * Syy - Sxy * Sxy
    trace = Sxx + Syy
    return det - k * trace * trace

def fast_corner_manual(image, threshold=20):
    img = image.astype(np.float32)
    h, w = img.shape
    corners = np.zeros((h, w), dtype=np.float32)
    circle_pts = [(0,-3),(1,-3),(2,-2),(3,-1),(3,0),(3,1),(2,2),(1,3),
                  (0,3),(-1,3),(-2,2),(-3,1),(-3,0),(-3,-1),(-2,-2),(-1,-3)]
    for y in range(3, h-3):
        for x in range(3, w-3):
            cv_val = img[y, x]
            bright = sum(1 for dx, dy in circle_pts if img[y+dy, x+dx] > cv_val + threshold)
            dark = sum(1 for dx, dy in circle_pts if img[y+dy, x+dx] < cv_val - threshold)
            corners[y, x] = max(bright, dark) if max(bright, dark) >= 12 else 0
    return corners

img = cv_imread('feature_image.jpg', cv2.IMREAD_GRAYSCALE)
harris_resp = harris_corner_manual(img)
harris_pts = harris_resp > 0.01 * harris_resp.max()
print(f'Harris: {harris_pts.sum()} corners')
fast_resp = fast_corner_manual(img, 20)
fast_pts = fast_resp > 10
print(f'FAST: {fast_pts.sum()} corners')
img_c = cv_imread('feature_image.jpg')
for y in range(img.shape[0]):
    for x in range(img.shape[1]):
        if harris_pts[y, x]:
            cv2.circle(img_c, (x, y), 3, (0,0,255), -1)
cv_imwrite('harris_manual.jpg', img_c)
print('角点检测完成！')



### 💻 代码要点解释

1. **数据准备**：加载测试图像，转换数据类型

2. **算法实现**：手写核心逻辑，逐步实现每个步骤

3. **对比验证**：与 OpenCV 对应函数结果进行数值对比

4. **结果可视化**：保存处理结果，观察效果差异

5. **扩展思考**：尝试不同参数，观察算法表现

---

</details>

---
